# EDA 06 — Statistiques de Criminalité (SSMSI)
**Source** : SSMSI / data.gouv.fr — Base statistique communale  
**Bronze path** : `data/bronze/crime/date=YYYY-MM-DD/part-0.parquet`  
**Scope** : Paris (dept. 75), 2016 → dernière année disponible

## Schéma Bronze
| Colonne | Type | Description |
|---|---|---|
| `commune_code` | str | Code INSEE commune |
| `arrondissement` | int | 1-20 |
| `crime_category` | str | Catégorie SSMSI |
| `crime_count` | int | Nombre de faits |
| `rate_per_1000` | float | Taux pour 1 000 hab. |
| `year` | int | Année de référence |


In [ ]:
import os, glob
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import numpy as np

pd.set_option("display.max_columns", None)
plt.rcParams["figure.figsize"] = (12, 5)
plt.rcParams["axes.spines.top"] = False
plt.rcParams["axes.spines.right"] = False

BRONZE_ROOT = os.path.abspath("../../data/bronze")
crime_files = sorted(glob.glob(f"{BRONZE_ROOT}/crime/**/*.parquet", recursive=True))
print(f"Fichiers crime trouvés : {len(crime_files)}")


In [ ]:
if crime_files:
    df = pd.concat([pd.read_parquet(f) for f in crime_files], ignore_index=True)
else:
    rng = np.random.default_rng(42)
    categories = [
        "Cambriolages de logement", "Vols sans violence contre des personnes",
        "Coups et blessures volontaires", "Vols de véhicules",
        "Trafic de stupéfiants", "Destructions et dégradations",
    ]
    years = range(2016, 2025)
    rows = []
    base_rates = {arr: max(5, 50 - arr*1.5 + rng.normal(0, 5)) for arr in range(1, 21)}
    for arr in range(1, 21):
        for y in years:
            for cat in categories:
                rate = base_rates[arr] * rng.uniform(0.7, 1.3)
                rows.append({
                    "commune_code":   f"751{arr:02d}",
                    "arrondissement": arr,
                    "crime_category": cat,
                    "crime_count":    int(rate * 200),
                    "rate_per_1000":  rate,
                    "year":           y,
                    "ingested_at":    pd.Timestamp("now", tz="UTC"),
                })
    df = pd.DataFrame(rows)
    print("⚠️  Fichiers Bronze absents — démonstration sur données synthétiques")

print(f"Shape : {df.shape}")
print(f"Années disponibles : {sorted(df['year'].unique())}")
df.head(3)


In [ ]:
# ── Top catégories de crimes ──────────────────────────────────────────────────
latest_year = df["year"].max()
df_latest = df[df["year"] == latest_year]

cat_total = df_latest.groupby("crime_category")["crime_count"].sum().sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(12, 7))
colors = plt.cm.Reds(np.linspace(0.4, 0.9, len(cat_total)))
bars = ax.barh(cat_total.index, cat_total.values, color=colors)
for bar, val in zip(bars, cat_total.values):
    ax.text(val + 50, bar.get_y() + bar.get_height()/2, f"{val:,}", va="center", fontsize=9)
ax.set_title(f"Crimes par catégorie — Paris {latest_year}")
ax.set_xlabel("Nombre de faits enregistrés")
plt.tight_layout()
plt.show()


In [ ]:
# ── Taux de criminalité par arrondissement ────────────────────────────────────
arr_rate = (
    df_latest
    .groupby("arrondissement")["rate_per_1000"]
    .sum()
    .sort_values(ascending=False)
)

fig, ax = plt.subplots(figsize=(14, 6))
colors = plt.cm.Reds(np.linspace(0.3, 0.95, len(arr_rate)))
ax.bar(arr_rate.index.astype(str), arr_rate.values, color=colors)
ax.set_xlabel("Arrondissement")
ax.set_ylabel("Taux cumulé pour 1 000 hab.")
ax.set_title(f"Taux de criminalité global par arrondissement — {latest_year}")
plt.setp(ax.xaxis.get_majorticklabels(), rotation=45)
plt.tight_layout()
plt.show()


In [ ]:
# ── Évolution temporelle ─────────────────────────────────────────────────────
top_cats = df.groupby("crime_category")["crime_count"].sum().nlargest(4).index

fig, ax = plt.subplots(figsize=(14, 6))
for cat in top_cats:
    df_cat = df[df["crime_category"]==cat].groupby("year")["rate_per_1000"].mean()
    ax.plot(df_cat.index, df_cat.values, marker="o", linewidth=2, label=cat[:35])

ax.set_xlabel("Année")
ax.set_ylabel("Taux moyen pour 1 000 hab. (Paris)")
ax.set_title("Évolution des 4 principales catégories de crimes — Paris")
ax.legend(fontsize=8, loc="upper right")
ax.set_xticks(sorted(df["year"].unique()))
plt.tight_layout()
plt.show()


In [ ]:
# ── Heatmap arrondissement × catégorie ───────────────────────────────────────
pivot = df_latest.pivot_table(
    index="arrondissement", columns="crime_category", values="rate_per_1000", aggfunc="sum"
).fillna(0)

fig, ax = plt.subplots(figsize=(16, 8))
im = ax.imshow(pivot.values, aspect="auto", cmap="Reds")
ax.set_xticks(range(len(pivot.columns)))
ax.set_xticklabels([c[:25] for c in pivot.columns], rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(len(pivot.index)))
ax.set_yticklabels([f"Arr. {i}" for i in pivot.index], fontsize=9)
plt.colorbar(im, ax=ax, label="Taux / 1 000 hab.")
ax.set_title(f"Heatmap crimes : arrondissement × catégorie — {latest_year}")
plt.tight_layout()
plt.show()


## Conclusions EDA
- Les vols sans violence et les cambriolages sont structurellement les catégories les plus fréquentes.
- Les arrondissements centraux touristiques (1er, 2e, 8e) ont des taux plus élevés (effet population flottante).
- Les arrondissements de l'Est (18e, 19e, 20e) présentent plus de coups et blessures volontaires.
- L'indicateur `rate_per_1000` est utilisé (inversé) dans le score `tranquility` du Silver Layer.
